43 by 43 matrix (0 to 39 in python) with 41,41,42 being 3 turns in Jail


Standard Distribution of 2 dice
|+2|+3|+4|+5|+6|+7|+8|+9|+10|+11|+12|
|--|--|--|--|--|--|--|--|--|--|--|
|1/36|2/36|3/36|4/36|5/36|6/36|5/36|4/36|3/36|2/36|1/36|

.

Purely for HOUSE JAIL RULES:
Distribution when we can't allow doubles:
|+2|+3|+4|+5|+6|+7|+8|+9|+10|+11|+12|
|--|--|--|--|--|--|--|--|--|--|--|
|0|2/30|2/30|4/30|4/30|6/30|4/30|4/30|2/30|2/30|0|

List of Monopoly Chance cards (UK)Board Games

1. Advance to Go (Collect £200)

2. Advance to Trafalgar Square. If you pass Go, collect £200

3. Advance to Mayfair

4. Advance to Pall Mall. If you pass Go, collect £200

5. Advance to the nearest Station. If unowned, you may buy it from the Bank. If owned, pay wonder twice the rental to which they are otherwise entitled.

6. Advance to the nearest Station. If unowned, you may buy it from the Bank. If owned, pay wonder twice the rental to which they are otherwise entitled.

7. Advance token to nearest Utility. If unowned, you may buy it from the Bank. If owned, throw dice and pay owner a total ten times amount thrown.

8. Bank pays you dividend of £50

9. Get Out of Jail Free

10. Go Back 3 Spaces

11. Go to Jail. Go directly to Jail, do not pass Go, do not collect £200

12. Make general repairs on all your property. For each house pay £25. For each hotel pay £100

13. Speeding fine £15

14. Take a trip to Kings Cross Station. If you pass Go, collect £200

15. You have been elected Chairman of the Board. Pay each player £50

16. Your building loan matures. Collect £150


Important: 1, 2, 3, 4, 5, 6, 7, 10, 11, 14

List of Monopoly Community Chest cards (UK)

1. Advance to Go (Collect £200)

2. Bank error in your favour. Collect £200

3. Doctor’s fee. Pay £50

4. From sale of stock you get £50

5. Get Out of Jail Free

6. Go to Jail. Go directly to jail, do not pass Go, do not collect £200

7. Holiday fund matures. Receive £100

8. Income tax refund. Collect £20

9. It is your birthday. Collect £10 from every player

10. Life insurance matures. Collect £100

11. Pay hospital fees of £100

12. Pay school fees of £50

13. Receive £25 consultancy fee

14. You are assessed for street repairs. £40 per house. £115 per hotel

15. You have won second prize in a beauty contest. Collect £10

16. You inherit £100


Important: 1, 6

In [89]:
import numpy as np
from numpy.linalg import eig
from scipy import sparse

size = 43 #Modelling indices 40, 41, 42 as the 3 stages of Jail
rows = []
cols = []
data = []

#30 = Go To Jail
#2, 17, 33 = Community
#7, 22, 36 = chance
special_indices = [30, 2, 7, 17, 22, 33, 36]

In [90]:
#fetching transition matrix probabilities
# IMPORTANT: We are constructing a Transposed Matrix, by switching the columns and rows - purely because (arr.T).dot(t) is faster at multiplying than t @ arr

#if we land on Chance 10): Move 3 steps back; this could potentially lead to positions 4, 18, 33.
# 33 is special index (community), which can cause Go or Jail
#Therefore we  need to do recursion when new_pos==36, new_pos-3 = 33

for i in range(40):
    for moves in range(2,13):

        #finding new position on board, accounting for looping around 0
        new_pos = i+moves
        if new_pos>39: new_pos=new_pos%40

        #calculating standard distribution probability
        if moves < 8: data_val = (moves-1)/36
        else: data_val = (13-moves)/36

        #ordinary movement, no special_indices
        if new_pos not in special_indices:
            cols.append(i)
            rows.append(new_pos)
            data.append(data_val) 

        # Now checking for special_indices.
        else:

            #Go To Jail
            if new_pos==30:
                cols.append(i)
                rows.append(40)
                data.append(data_val)

            #chance
            elif new_pos in [7, 22, 36]:

                # 10) Moving 3 steps back, P = 1/16
                n = new_pos - 3
                if n <0: n+=40
                ########checking for recursion here: #####################################
                if n==33:
                    #recursion true. We go to community. Do all community stuff here
                    cols.append(i)
                    rows.append(n)
                    data.append((1/16)*data_val * (14/16))  #14/16 chance of remaining in current position

                    cols.append(i)
                    rows.append(0)
                    data.append((1/16)*data_val * (1/16) ) # 1) 1/16 chance of going to Go [0]

                    cols.append(i)
                    rows.append(40)
                    data.append((1/16)*data_val * (1/16)) # 6)  1/16 chance of going to Jail [40]
                else:
                    cols.append(i)
                    rows.append(n)
                    data.append(data_val * (1/16))
                ###########################################################################


                #6/16 chance of remaining in current position
                cols.append(i)
                rows.append(new_pos)
                data.append( data_val * (6/16))

                # 1)  1/16 for going to Go [0]
                cols.append(i)
                rows.append(0)
                data.append( data_val * (1/16))

                # 2) 1/16 going to Trafalgar square [24]
                cols.append(i)
                rows.append(24)
                data.append(data_val * (1/16))

                # 3) 1/16 going to Mayfair [39]
                cols.append(i)
                rows.append(39)
                data.append(data_val * (1/16))      

                # 4) 1/16 to Pall Mall [11]
                cols.append(i)
                rows.append(11)
                data.append(data_val * (1/16))     

                # 5) and 6) are going to nearest Station. Therefore P = 2/16. There are Stations at positions [5, 15, 25, 35]
                cols.append(i)
                if new_pos%10 >5: rail_loc = new_pos + 15 - (new_pos%10)
                else: rail_loc = new_pos + (5 - (new_pos%10))
                if rail_loc > 40: rail_loc-=40
                rows.append(rail_loc)
                data.append (data_val * (2/16))

                # 7) advance to nearest utility. P = 1/16. Utilities at [12, 28]
                cols.append(i)
                data.append( data_val * (1/16))
                if new_pos==22: rows.append(28)
                else: rows.append(12) #when at 7 or 36, will go to 12

                # 11) 1/16 for going to Jail [40]
                cols.append(i)
                rows.append(40)
                data.append(data_val * (1/16))

                #14) 1/16 for going to Railway Station at [5]
                cols.append(i)
                rows.append(5)
                data.append(data_val * (1/16))
        
            
            #Community
            elif new_pos in [2, 17, 33]:
                cols.append(i)
                rows.append(new_pos)
                data.append(data_val * (14/16))  #14/16 chance of remaining in current position

                cols.append(i)
                rows.append(0)
                data.append(data_val * (1/16) ) # 1) 1/16 chance of going to Go [0]

                cols.append(i)
                rows.append(40)
                data.append(data_val * (1/16)) # 6)  1/16 chance of going to Jail [40]


In [ ]:
#Jail setups with size 43, and jails 40, 41, 42
#Choose House or Standard Jail Rules
rules = "Standard"

#HOUSE JAIL RULES!!!!
if rules=="House":
    for moves in range(2, 13):
        if moves < 8:
            standard_data_val = (moves-1) / 36
        else:
            standard_data_val = (13-moves) / 36

        #Getting out of Jail 1 with a double and then second roll for next position
        cols.append(40)
        rows.append(10+moves)
        data.append( (1/6) * standard_data_val)

        #Getting out of Jail 2 with a double, and then second roll for next position
        cols.append(41)
        rows.append(10+moves)
        data.append( (1/6) * standard_data_val)

    #Moving from J1 to J2
    cols.append(40)
    rows.append(41)
    data.append(5/6)

    #Moving from J2 to J3
    cols.append(41)
    rows.append(42)
    data.append(5/6)

    #Getting out of J3:
    for locations in range(2, 13):
        cols.append(42)
        rows.append(10+locations)

        if locations==2 or locations==12: # If we move 2 or 12 positions from Jail, We know there was a double roll to get out of Jail 3, as otherwise a 1,1 or 6,6 movement wouldn't be possible
            data.append(1/6 * 1/36) #the 1/6 is from doubles, the 1/36 for the second roll being either 2 or 12
    
        elif locations==4 or locations==3:
            data.append((1/6)*((locations-1)/36) + (5/6)*(2/30)) # double_roll * standard_second_roll  + undouble roll AND sum= 3 or4
    
        elif locations ==6 or locations==5:
            data.append((1/6)*((locations-1)/36) + (5/6)*(4/30)) # double_roll * standard_second_roll  + undouble roll AND being sum= 5 or 6
    
        elif locations == 10 or locations==11:
            data.append((1/6)*((13-locations)/36) + (5/6)*(2/30)) # double_roll * standard_second_roll  + undouble roll AND being sum= 10 or 11
    
        elif locations ==8 or locations==9:
            data.append((1/6)*((13-locations)/36) + (5/6)*(4/30)) # double_roll * standard_second_roll  + undouble roll AND being sum=8 or 9
    
        else:
            data.append((1/6)*(6/36) + (5/6)*(6/30)) # double_roll * standard_second_roll  + undouble roll AND being sum= 7



# STANDARD JAIL RULES:
else:
    rules = "Standard"
    for moves in [2, 4, 6, 8, 10, 12]:
        #Getting out of Jail 1 with a double, and move forwards that amount. Therefore probs of going forwards 2... 12 = 1/36
        cols.append(40)
        rows.append(10+moves)
        data.append(1/36)

        #Getting out of Jail 2 with a double, and move forwards that amount
        cols.append(41)
        rows.append(10+moves)
        data.append(1/36)

    #Moving from J1 to J2
    cols.append(40)
    rows.append(41)
    data.append(5/6)

    #Moving from J2 to J3
    cols.append(41)
    rows.append(42)
    data.append(5/6)

    #Getting out of J3 regardless of a double; so you go forwards that amount.
    for moves in range(2, 13):
        if moves < 8: data_val = (moves-1)/36
        else: data_val = (13-moves)/36
        cols.append(42)
        rows.append(10+moves)
        data.append(data_val)

In [97]:
arr = sparse.csr_matrix((data, (rows, cols)), shape=(size, size))

#checking our Column-Stochastic matrix has columns which actually sum to one
col_sums = np.array(arr.sum(axis=0)).flatten()
print(np.min(col_sums), np.max(col_sums))


#picking matrix iteration, or eigendecomposition methods for calculating stationary dist:
method = "eigen"


######### MATRIX ITERATION METHOD ####################################################################################
# intial marginal distribution t. Can be anything, but I just chose a vector with equal distribution to start
# marginal distribution t should converge to stationary distribution t, over time.
if method == "iteration":
    s = np.ones(size) / size
    for i in range(300):
        new_s = arr.dot(s)

        # convergence detection
        if np.linalg.norm(new_s -s, 1) <= 1e-10:
            print(f"Broken at {i}")
            break
        s = new_s

######## EIGENDECOMPOSITION METHOD #####################################################################################
'''Find eigendecompositions: Whenever eigenvalue==1, we go from (Matrix * vector == eigenvalue * vector) to (Matrix * vector == vector)
For MCs, we have s*P = s, with row stochasticity
And, we have P*s = s, with column stochasticity
We constructed our matrix with column stochasticity, so our life is made easier because we only need to find right eigenvector = stationary_dist (t)'''

if method =="eigen":
    eigenvalues, eigenvectors = np.linalg.eig(arr.toarray())

    #finding the index of the eigenvalue = 1, to get corresponding eigenvector = stationary distribution
    idx = np.argmin(np.abs(eigenvalues-1.0))
    s = (eigenvectors[:, idx]).real

    #normalising stationary distribution
    s = s / s.sum()
#########################################################################################################################


print(np.sum(s))  # should be ~1

0.9999999999999998 1.0000000000000002
0.9999999999999999


In [106]:
# Showing top monopoly locations:
board_london = ["GO","Old Kent Road","Community Chest","Whitechapel Road","Income Tax","King's Cross Station","The Angel Islington","Chance","Euston Road","Pentonville Road","Jail (Just Visiting)","Pall Mall","Electric Company","Whitehall","Northumberland Avenue","Marylebone Station","Bow Street","Community Chest","Marlborough Street","Vine Street","Free Parking","Strand","Chance","Fleet Street","Trafalgar Square","Fenchurch Street Station","Leicester Square","Coventry Street","Water Works","Piccadilly","Go To Jail","Regent Street","Oxford Street","Community Chest","Bond Street","Liverpool Street Station","Chance","Park Lane","Super Tax","Mayfair","Jail 1","Jail 2","Jail 3"]
board_ny = ["GO","Mediterranean Avenue","Community Chest","Baltic Avenue","Income Tax","Reading Railroad","Oriental Avenue","Chance","Vermont Avenue","Connecticut Avenue","Jail (Just Visiting)","St. Charles Place","Electric Company","States Avenue","Virginia Avenue","Pennsylvania Railroad","St. James Place","Community Chest","Tennessee Avenue","New York Avenue","Free Parking","Kentucky Avenue","Chance","Indiana Avenue","Illinois Avenue","B&O Railroad","Atlantic Avenue","Ventnor Avenue","Water Works","Marvin Gardens","Go To Jail","Pacific Avenue","North Carolina Avenue","Community Chest","Pennsylvania Avenue","Short Line Railroad","Chance","Park Place","Luxury Tax","Boardwalk","Jail 1","Jail 2","Jail 3"]
board_india = ["GO", "Guwahati", "Community Chest", "Bhubaneshwar", "Income Tax", "Chennai Central Railway Station", "Panaji (Goa)", "Chance", "Agra", "Vadodara", "Jail (Just Visiting)", "Ludhiana", "Electric Company", "Patna", "Bhopal", "Howrah Railway Station", "Indore", "Community Chest", "Nagpur", "Kochi", "Free Parking", "Lucknow", "Chance", "Chandigarh", "Jaipur", "New Delhi Railway Station", "Pune", "Hyderabad", "Water Company", "Ahmedabad", "Go To Jail", "Kolkata", "Chennai", "Community Chest", "Bengalaru", "Chatrapati Shivaji Terminus", "Chance", "Delhi", "Super Tax", "Mumbai", "Jail 1", "Jail 2", "Jail 3"]

top_matrix_indices = np.argsort(s)[-10:][::-1]

print("The most visited locations - not including Jail, Go, Free Parking, Tax, Community, Chance \n")
no = [40, 41, 42, 0, 2, 38, 20, 30, 2, 7, 17, 22, 33, 36]

print("Board London: ")
for i in top_matrix_indices:
    if i not in no: print(f"{i}: \t {board_london[i]}")

print("\nBoard India: ")
for i in top_matrix_indices:
    if i not in no: print(f"{i}: \t {board_india[i]}")

print("\nBoard New York: ")
for i in top_matrix_indices:
    if i not in no: print(f"{i}: \t {board_ny[i]}")


The most visited locations - not including Jail, Go, Free Parking, Tax, Community, Chance 

Board London: 
24: 	 Trafalgar Square
25: 	 Fenchurch Street Station
5: 	 King's Cross Station
18: 	 Marlborough Street
19: 	 Vine Street
28: 	 Water Works

Board India: 
24: 	 Jaipur
25: 	 New Delhi Railway Station
5: 	 Chennai Central Railway Station
18: 	 Nagpur
19: 	 Kochi
28: 	 Water Company

Board New York: 
24: 	 Illinois Avenue
25: 	 B&O Railroad
5: 	 Reading Railroad
18: 	 Tennessee Avenue
19: 	 New York Avenue
28: 	 Water Works


In [116]:
groups = [
    ["Brown", [1, 3], 0],
    ["Light Blue", [6, 8, 9], 0],
    ["Pink", [11, 13, 14], 0],
    ["Orange", [16, 18, 19], 0],
    ["Red", [21, 23, 24], 0],
    ["Yellow", [26, 27, 29], 0],
    ["Green", [31, 32, 34], 0],
    ["Dark Blue", [37, 39], 0],
    ["Railway Stations", [5, 15, 25, 35], 0],
    ["Utilities", [12, 28], 0],
]

for g in groups:
    indices = g[1]
    combined_prob = 0
    for i in indices: combined_prob+=float(s[i])
    combined_prob = combined_prob / len(indices)
    g[2] = round(combined_prob*100, 5)

groups.sort(key=lambda g: g[2], reverse=True)

print(f"With {rules} Rules and {method.upper()} Method of calculating the stationary dist,\nhere are property groups ranked best to worst, by mean percentage probability of landing on a location in said group.\n")
for g in groups: print(f"{g[0]:<18}:   {g[2]}")

With Standard Rules and EIGEN Method of calculating the stationary dist,
here are property groups ranked best to worst, by mean percentage probability of landing on a location in said group.

Orange            :   2.75432
Red               :   2.7344
Railway Stations  :   2.6699
Utilities         :   2.64942
Yellow            :   2.52583
Green             :   2.48144
Pink              :   2.40231
Dark Blue         :   2.30031
Light Blue        :   2.19678
Brown             :   2.05882
